In [6]:
import pandas as pd
import numpy as np

print("Pandas version:", pd.__version__)
print("NumPy version:", np.__version__)
print("Notebook is working!")

Pandas version: 2.3.3
NumPy version: 2.2.6
Notebook is working!


In [7]:
import pandas as pd

basics_path = "../data/raw/title.basics.tsv"
ratings_path = "../data/raw/title.ratings.tsv"

basic_columns = [
    "tconst",
    "titleType",
    "primaryTitle",
    "startYear",
    "runtimeMinutes",
    "genres"
]

rating_columns = [
    "tconst",
    "averageRating",
    "numVotes"
]

basics = pd.read_csv(
    basics_path,
    sep="\t",
    na_values="\\N",
    usecols=basic_columns,
    low_memory=False
)

ratings = pd.read_csv(
    ratings_path,
    sep="\t",
    na_values="\\N",
    usecols=rating_columns
)

print("Basics shape:", basics.shape)
print("Ratings shape:", ratings.shape)

Basics shape: (12732739, 6)
Ratings shape: (1707914, 3)


In [8]:
basics.head()

,tconst,titleType,primaryTitle,startYear,runtimeMinutes,genres
0,tt0000001,short,Carmencita,1894.0,1,"Documentary,Short"
1,tt0000002,short,Le clown et ses chiens,1892.0,5,"Animation,Short"
2,tt0000003,short,Poor Pierrot,1892.0,5,"Animation,Comedy,Romance"
3,tt0000004,short,Un bon bock,1892.0,12,"Animation,Short"
4,tt0000005,short,Blacksmith Scene,1893.0,1,Short


In [9]:
ratings.head()

,tconst,averageRating,numVotes
0,tt0000001,5.7,2226
1,tt0000002,5.4,324
2,tt0000003,6.4,2372
3,tt0000004,5.0,201
4,tt0000005,6.2,3087


In [10]:
basics.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12732739 entries, 0 to 12732738
Data columns (total 6 columns):
 #   Column          Dtype  
---  ------          -----  
 0   tconst          object 
 1   titleType       object 
 2   primaryTitle    object 
 3   startYear       float64
 4   runtimeMinutes  object 
 5   genres          object 
dtypes: float64(1), object(5)
memory usage: 582.9+ MB


In [11]:
movies = basics[basics["titleType"] == "movie"].copy()

print("Total movie records:", len(movies))

Total movie records: 754810


In [12]:
movies.head()

,tconst,titleType,primaryTitle,startYear,runtimeMinutes,genres
8,tt0000009,movie,Miss Jerry,1894.0,45,Romance
144,tt0000147,movie,The Corbett-Fitzsimmons Fight,1897.0,100,"Documentary,News,Sport"
498,tt0000502,movie,Bohemios,1905.0,100,NaN
570,tt0000574,movie,The Story of the Kelly Gang,1906.0,70,"Action,Adventure,Biography"
587,tt0000591,movie,The Prodigal Son,1907.0,90,Drama


In [13]:
movies["startYear"] = pd.to_numeric(
    movies["startYear"],
    errors="coerce"
)

In [14]:
movies = movies.dropna(subset=["startYear"])

In [15]:
movies["startYear"] = movies["startYear"].astype(int)

In [16]:
movies["startYear"].describe()

count    641483.000000
mean       1995.197757
std          30.060584
min        1894.000000
25%        1978.000000
50%        2008.000000
75%        2018.000000
max        2032.000000
Name: startYear, dtype: float64

In [17]:
movies = movies[
    (movies["startYear"] >= 1900) &
    (movies["startYear"] <= 2026)
].copy()

print("Movies after year filtering:", len(movies))

Movies after year filtering: 640566


In [18]:
movies["startYear"].describe()

count    640566.000000
mean       1995.155616
std          30.053783
min        1900.000000
25%        1978.000000
50%        2008.000000
75%        2018.000000
max        2026.000000
Name: startYear, dtype: float64

In [19]:
print(movies["startYear"].min())
print(movies["startYear"].max())

1900
2026


In [20]:
movies["runtimeMinutes"] = pd.to_numeric(
    movies["runtimeMinutes"],
    errors="coerce"
)

In [21]:
movies["runtimeMinutes"].describe()

count    468070.000000
mean         89.859519
std         115.443498
min           1.000000
25%          73.000000
50%          89.000000
75%         100.000000
max       51420.000000
Name: runtimeMinutes, dtype: float64

In [22]:
print("Missing genres before cleaning:", movies["genres"].isna().sum())
movies = movies.dropna(subset=["genres"]).copy()

print("Missing genres after cleaning:", movies["genres"].isna().sum())
print("Movies remaining:", len(movies))

Missing genres before cleaning: 70908
Missing genres after cleaning: 0
Movies remaining: 569658


In [23]:
movies["genres"].head(20)

570     Action,Adventure,Biography
587                          Drama
610                          Drama
625                          Drama
668                          Drama
672              Adventure,Fantasy
876                          Drama
930                          Drama
1016                        Comedy
1037                     Drama,War
1047                   Documentary
1100                         Drama
1103                         Crime
1135                   Documentary
1151                   Documentary
1163                 Drama,Romance
1172               Adventure,Drama
1228                         Drama
1265                         Drama
1273        Biography,Drama,Family
Name: genres, dtype: object

In [24]:
movies["genre_list"] = movies["genres"].str.split(",")
movies[
    ["primaryTitle", "genres", "genre_list"]
].head(10)

,primaryTitle,genres,genre_list
570,The Story of the Kelly Gang,"Action,Adventure,Biography","[Action, Adventure, Biography]"
587,The Prodigal Son,Drama,[Drama]
610,Robbery Under Arms,Drama,[Drama]
625,Hamlet,Drama,[Drama]
668,Don Quijote,Drama,[Drama]
672,The Fairylogue and Radio-Plays,"Adventure,Fantasy","[Adventure, Fantasy]"
876,"Hamlet, Prince of Denmark",Drama,[Drama]
930,Locura de amor,Drama,[Drama]
1016,Salome Mad,Comedy,[Comedy]
1037,Gøngehøvdingen,"Drama,War","[Drama, War]"


In [25]:
genre_df = movies.explode("genre_list").copy()
genre_df = genre_df.rename(
    columns={"genre_list": "genre"}
)
genre_df[
    ["tconst", "primaryTitle", "startYear", "genre"]
].head(20)

,tconst,primaryTitle,startYear,genre
570,tt0000574,The Story of the Kelly Gang,1906,Action
570,tt0000574,The Story of the Kelly Gang,1906,Adventure
570,tt0000574,The Story of the Kelly Gang,1906,Biography
587,tt0000591,The Prodigal Son,1907,Drama
610,tt0000615,Robbery Under Arms,1907,Drama
625,tt0000630,Hamlet,1908,Drama
668,tt0000675,Don Quijote,1908,Drama
672,tt0000679,The Fairylogue and Radio-Plays,1908,Adventure
672,tt0000679,The Fairylogue and Radio-Plays,1908,Fantasy
876,tt0000886,"Hamlet, Prince of Denmark",1910,Drama


In [26]:
print("Number of unique genres:", genre_df["genre"].nunique())

print("\nGenre counts:")
print(genre_df["genre"].value_counts())

Number of unique genres: 27

Genre counts:
genre
Drama          240102
Documentary    139858
Comedy         109361
Action          49705
Romance         49304
Crime           38141
Thriller        37978
Horror          34078
Adventure       26762
Family          17923
Mystery         17632
Biography       17011
Music           15572
History         14857
Fantasy         14549
Sci-Fi          10707
Musical         10346
Animation        9187
War              9144
Adult            9016
Sport            8171
Western          7195
News             1431
Film-Noir         876
Reality-TV        529
Talk-Show         204
Game-Show           7
Name: count, dtype: int64


In [27]:
print("Genres containing '_':")

print(
    genre_df.loc[
        genre_df["genre"].str.contains("_", na=False),
        "genre"
    ].value_counts()
)

Genres containing '_':
Series([], Name: count, dtype: int64)


In [28]:
duplicate_pairs = genre_df[
    ["tconst", "genre"]
].duplicated().sum()

print("Duplicate movie-genre pairs:", duplicate_pairs)

Duplicate movie-genre pairs: 0


In [29]:
duplicate_movies = movies["tconst"].duplicated().sum()

print("Duplicate movie IDs:", duplicate_movies)

Duplicate movie IDs: 0


In [30]:
print("Runtime less than 1 minute:",
      (movies["runtimeMinutes"] < 1).sum())

print("Runtime greater than 400 minutes:",
      (movies["runtimeMinutes"] > 400).sum())

Runtime less than 1 minute: 0
Runtime greater than 400 minutes: 213


In [31]:
print("Missing values in movies:")
print(movies.isna().sum())

Missing values in movies:
tconst                 0
titleType              0
primaryTitle           3
startYear              0
runtimeMinutes    128024
genres                 0
genre_list             0
dtype: int64


In [32]:
long_movies = movies[
    movies["runtimeMinutes"] > 400
].copy()

print("Movies with runtime > 400 minutes:", len(long_movies))

long_movies[
    ["tconst", "primaryTitle", "startYear", "runtimeMinutes", "genres"]
].sort_values(
    "runtimeMinutes",
    ascending=False
).head(20)

Movies with runtime > 400 minutes: 213


,tconst,primaryTitle,startYear,runtimeMinutes,genres
11979263,tt8273150,Logistics,2012,51420.0,Documentary
9197288,tt3854496,Ambiancé,2020,43200.0,Documentary
2222205,tt12277054,Carnets Filmés (Liste Complète),2019,28643.0,Documentary
6291335,tt2659636,Modern Times Forever,2011,14400.0,Documentary
5926529,tt2355497,Beijing 2003,2004,9000.0,Documentary
10543683,tt5068890,Hunger!,2015,6000.0,"Documentary,Drama"
4715236,tt1806963,Matrjoschka,2006,5700.0,Drama
10573260,tt5136218,London EC1,2015,5460.0,"Comedy,Drama,Mystery"
271673,tt0284020,The Cure for Insomnia,1987,5220.0,"Documentary,Music"
9171299,tt3837350,A 2nd generation film,2013,3077.0,Drama


In [36]:
movies[
    movies["primaryTitle"].isna()
]
movies = movies.dropna(
    subset=["primaryTitle"]
).copy()

print(
    "Missing titles after cleaning:",
    movies["primaryTitle"].isna().sum()
)

Missing titles after cleaning: 0


In [37]:
movies.loc[
    movies["runtimeMinutes"] > 400,
    "runtimeMinutes"
] = np.nan

print(
    "Runtime values > 400 after cleaning:",
    (movies["runtimeMinutes"] > 400).sum()
)

Runtime values > 400 after cleaning: 0


In [38]:
print("Minimum runtime:",
      movies["runtimeMinutes"].min())

print("Maximum runtime:",
      movies["runtimeMinutes"].max())

print("Missing runtime values:",
      movies["runtimeMinutes"].isna().sum())

Minimum runtime: 1.0
Maximum runtime: 400.0
Missing runtime values: 128235


In [39]:
print("========== FINAL CLEANING CHECK ==========")

print("Total movies:", len(movies))

print(
    "Duplicate movie IDs:",
    movies["tconst"].duplicated().sum()
)

print(
    "Missing titles:",
    movies["primaryTitle"].isna().sum()
)

print(
    "Missing years:",
    movies["startYear"].isna().sum()
)

print(
    "Missing genres:",
    movies["genres"].isna().sum()
)

print(
    "Missing runtime:",
    movies["runtimeMinutes"].isna().sum()
)

print(
    "Invalid runtime > 400:",
    (movies["runtimeMinutes"] > 400).sum()
)

========== FINAL CLEANING CHECK ==========
Total movies: 569655
Duplicate movie IDs: 0
Missing titles: 0
Missing years: 0
Missing genres: 0
Missing runtime: 128235
Invalid runtime > 400: 0


In [40]:
# Recreate genre-level dataframe from cleaned movies

genre_df = movies.explode("genre_list").copy()

genre_df = genre_df.rename(
    columns={"genre_list": "genre"}
)

print("Genre dataframe shape:", genre_df.shape)
print("Unique genres:", genre_df["genre"].nunique())
print("Missing genres:", genre_df["genre"].isna().sum())

Genre dataframe shape: (889641, 7)
Unique genres: 27
Missing genres: 0
